# FIREWATCH AI — Ingestion and Normalization

**Stage 04.** Run the auditable ingestion pipeline against the local 2025/2026 FIRMS CSVs. The pipeline leaves source files untouched, preserves raw text columns, adds normalized fields and quality flags, and writes Parquet plus a summary under `data/processed/`.

Run cells from top to bottom. Product/source semantics and NRT/archive lineage are documented in `reports/03_product_documentation_review.md` and `reports/DATA_CONTRACT.md`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / '2025').is_dir() and (p / '2026').is_dir():
            return p
    raise FileNotFoundError('Could not locate project root containing 2025/ and 2026/')

ROOT = find_root()
sys.path.insert(0, str(ROOT))
from src.firewatch.data_pipeline import discover_sources, normalize_source, ingest_workspace
OUTPUT = ROOT / 'data' / 'processed' / 'observations.parquet'
print('Project root:', ROOT)
print('Output:', OUTPUT)

## Discover source snapshots

The pipeline fingerprints each source file before processing. Keep these checksums in any exported report so results can be traced to exact inputs.

In [ ]:
sources = discover_sources(ROOT)
inventory = pd.DataFrame([{
    'year': s.year_folder, 'product': s.product_code, 'stream': s.stream_type,
    'file': s.relative_path, 'sha256': s.source_sha256
} for s in sources])
display(inventory[['year', 'product', 'stream', 'file']])
print(f'{len(sources)} input CSV files discovered')

## Normalize all files and save derived data

No rows are dropped. Invalid records receive `validation_status='quarantine'` and explanatory `quality_flags`; inspect them before deciding whether an analytical view should exclude them.

In [ ]:
observations, file_summary = ingest_workspace(ROOT, OUTPUT)
display(file_summary)
print('Normalized rows:', f'{len(observations):,}')
print('Parquet saved:', OUTPUT)
print('File-level summary saved:', OUTPUT.with_name('ingestion_summary.csv'))

## Data-quality results

Compare valid and quarantined rows by source. A quarantine flag preserves a row for audit while preventing silent use of invalid coordinates or timestamps.

In [ ]:
quality = (observations.groupby(['year_folder', 'product_code', 'stream_type', 'validation_status'])
           .size().rename('records').reset_index())
display(quality)
flag_counts = (observations.loc[observations.quality_flags.ne(''), 'quality_flags']
               .str.split(';').explode().value_counts().rename_axis('quality_flag').reset_index(name='records'))
display(flag_counts) if not flag_counts.empty else print('No validation flags found.')

## Visualize observation activity

These plots count records by the satellite product as delivered. They are not normalized fire-event counts and do not imply sensor-neutral comparability.

In [ ]:
monthly = (observations.dropna(subset=['observation_datetime_utc'])
           .assign(month=lambda d: d.observation_datetime_utc.dt.strftime('%Y-%m'))
           .groupby(['month', 'product_code']).size().unstack(fill_value=0).sort_index())
ax = monthly.plot(kind='bar', stacked=True, figsize=(13, 5), width=0.85)
ax.set_title('Delivered FIRMS observations by month and product')
ax.set_xlabel('UTC month'); ax.set_ylabel('Observation rows')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
display(monthly.tail(15))

In [ ]:
plot_data = observations.dropna(subset=['longitude_num', 'latitude_num'])
if len(plot_data) > 100_000:
    plot_data = plot_data.sample(100_000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 7))
for product_code, group in plot_data.groupby('product_code'):
    ax.scatter(group.longitude_num, group.latitude_num, s=2, alpha=0.28, label=product_code)
ax.set_title('Sample of delivered detection coordinates by product')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.legend(markerscale=4)
ax.grid(alpha=0.2); plt.tight_layout(); plt.show()
print('Point cloud only; no country boundary or claim of geographic completeness is implied.')

## Stage handoff

Before event association, review file-level coverage and quality results. The next notebook will evaluate spatial-temporal candidate associations and threshold sensitivity while keeping raw observation counts separate from inferred clusters.